# Notebook 11 – Handling Imbalanced Data
## What is Class Imbalance?

Class imbalance happens when one outcome in your target variable shows up
far more often than the other. Our `Attrition` column is a real example:
**1,001 employees stayed (No), only 229 left (Yes)**, roughly 81% versus 19%.

## Balanced vs Imbalanced Dataset

A balanced dataset would have roughly equal counts of each class, like
50/50 or 60/40. Ours is imbalanced, nearly 4 out of 5 employees are in the
"No" class. This matters a lot for how we train and evaluate a model.

In [3]:
import pandas as pd
import glob
import os

# Automatically locate the CSV, even if it's not in the current working directory
matches = glob.glob("**/hr_employee_attrition_raw.csv", recursive=True)

if matches:
    file_path = matches[0]
    print("Found file at:", file_path)
else:
    raise FileNotFoundError(
        "Could not find 'hr_employee_attrition_raw.csv' anywhere under: "
        + os.getcwd()
        + "\nMake sure the file is saved somewhere inside this folder or a subfolder."
    )

df = pd.read_csv(file_path)

print(df["Attrition"].value_counts())
print(df["Attrition"].value_counts(normalize=True).round(3))

Found file at: Downloads\hr_employee_attrition_raw.csv
Attrition
No     1001
Yes     229
Name: count, dtype: int64
Attrition
No     0.814
Yes    0.186
Name: proportion, dtype: float64


## Why Accuracy Alone Can Be Misleading

Here's the core problem, spelled out with our actual numbers.

Imagine a lazy model that predicts **"No" for every single employee**,
never actually learning anything about who leaves and who stays.

* Correct predictions: all 1,001 employees who stayed.
* Wrong predictions: all 229 employees who left, completely missed.
* **Accuracy: 1,001 / 1,230 = 81.4%**

That model looks like it's performing well by accuracy alone, 81.4% sounds
solid, but it has learned **nothing useful**. It cannot identify a single
employee at risk of leaving, which is the entire point of an attrition
model. Accuracy hides this failure completely because it treats both
classes as equally important, when the minority class ("Yes") is usually
the one that actually matters for business decisions.

In [4]:
# Demonstrate the "predict everything as majority class" trap
majority_class_accuracy = (df["Attrition"] == "No").mean()
print(f"Accuracy from predicting 'No' for everyone: {majority_class_accuracy:.1%}")
print("But this model correctly identifies 0 employees who actually left.")

Accuracy from predicting 'No' for everyone: 81.4%
But this model correctly identifies 0 employees who actually left.


## Undersampling

**What it does:** removes rows from the majority class ("No") until both
classes are closer in size.

**Trade-off:** simple and fast, but throws away real data. Reducing our
1,001 "No" rows down to 229 (to match "Yes") means discarding over 770
employees' worth of information, information that might have been useful.

In [5]:
from sklearn.utils import resample

df_majority = df[df["Attrition"] == "No"]
df_minority = df[df["Attrition"] == "Yes"]

df_majority_downsampled = resample(
    df_majority, replace=False, n_samples=len(df_minority), random_state=42
)

df_undersampled = pd.concat([df_majority_downsampled, df_minority])
print(df_undersampled["Attrition"].value_counts())

Attrition
No     229
Yes    229
Name: count, dtype: int64


## Random Oversampling

**What it does:** duplicates existing rows from the minority class ("Yes")
until it matches the majority class in count.

**Trade-off:** keeps all original data, but the duplicated rows are exact
copies. A model can start memorizing those specific 229 employees rather
than learning the general pattern behind attrition, increasing overfitting
risk.

In [6]:
df_minority_upsampled = resample(
    df_minority, replace=True, n_samples=len(df_majority), random_state=42
)

df_oversampled = pd.concat([df_majority, df_minority_upsampled])
print(df_oversampled["Attrition"].value_counts())

Attrition
No     1001
Yes    1001
Name: count, dtype: int64


## SMOTE (Synthetic Minority Oversampling Technique)

**What it does:** instead of duplicating existing minority rows, it
generates **new, synthetic** minority examples by interpolating between
real minority data points and their nearest neighbors.

**Why it's usually better than random oversampling:** the model sees
variety instead of exact duplicates, which reduces overfitting to specific
rows while still balancing the classes.

**Requirement:** SMOTE only works on numeric features, categorical columns
need to be encoded first (see Notebook 7).

In [11]:
!pip install imbalanced-learn


   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]




[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from imblearn.over_sampling import SMOTE

model_df = df[["Age", "MonthlyIncome", "JobSatisfaction", "PerformanceRating",
                "DistanceFromHomeKM", "Attrition"]].copy()
model_df["MonthlyIncome"] = pd.to_numeric(
    model_df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False), errors="coerce"
)
model_df = model_df.dropna()

X = model_df.drop(columns=["Attrition"])
y = model_df["Attrition"]

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
print(y_resampled.value_counts())

Attrition
No     899
Yes    899
Name: count, dtype: int64


## Borderline-SMOTE

**What it does:** a variation of SMOTE that focuses specifically on
minority samples near the decision boundary (the "borderline" cases that
are hardest to classify correctly), rather than generating synthetic
points uniformly across the whole minority class.

**When to prefer it over standard SMOTE:** when misclassifications tend to
happen near the boundary between classes, generating more examples there
gives the model extra help exactly where it struggles most, instead of
spreading synthetic data evenly where it may not be needed.

## Class Weights

**What it does:** instead of changing the dataset at all, this tells the
model to **penalize mistakes on the minority class more heavily** during
training. No new rows are created, no rows are removed.

**Why this is often the safest option:** it avoids the two core problems
of resampling entirely, no data is thrown away (unlike undersampling), and
no duplicate or synthetic rows are introduced (unlike oversampling/SMOTE).
The trade-off is that it requires the algorithm to support a `class_weight`
parameter, not all of them do.

In [13]:
from sklearn.linear_model import LogisticRegression

# class_weight="balanced" automatically weights classes inversely to their frequency
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X, y.map({"Yes": 1, "No": 0}))
print("Class weights automatically applied based on frequency.")

Class weights automatically applied based on frequency.


## When to Use Each Technique

* **Undersampling:** dataset is large enough that losing majority-class
  rows won't hurt much, and training speed matters. Not ideal for our
  1,230-row dataset, we'd be throwing away over 60% of it.
* **Random Oversampling:** quick fix when you need balance fast and
  overfitting risk is acceptable or will be monitored closely.
* **SMOTE:** solid default for numeric, well-behaved features when you
  want variety instead of duplicate rows.
* **Borderline-SMOTE:** when misclassification specifically happens near
  the class boundary, worth trying when standard SMOTE doesn't improve
  recall enough.
* **Class Weights:** often the best first attempt, since it changes
  nothing about the data itself, just how errors are penalized during
  training. Try this before reaching for resampling methods.